In [1]:
import os

base_path = "/Users/mihokoda/Desktop/LocationReasoner/dubai_res"
difficulties = ["sim", "med", "hard"]

for difficulty in difficulties:
    parent_folder = os.path.join(base_path, difficulty)
    for i in range(10):
        subfolder_name = f"tc_dubai_res_{difficulty}_{i}"
        subfolder_path = os.path.join(parent_folder, subfolder_name)
        os.makedirs(subfolder_path, exist_ok=True)
        print(f"Created: {subfolder_path}")


Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_0
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_1
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_2
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_3
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_4
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_5
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_6
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_7
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_8
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_9
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/med/tc_dubai_res_med_0
Created: /Users/mihokoda/Desktop/LocationReasoner/dubai_res/med/tc_dubai_res_med_1
Crea

In [ ]:
import pandas as pd

path = "/Users/mihokoda/Desktop/City LLM final analysis/book1.xlsx"
df = pd.read_excel(path, sheet_name="dubai")

llm_name = ["gemini1.5", "gemini2.5", "react", "react + reflexion"]

for curr_llm in llm_name:
    1. only consider if df['llm_name'] == curr_llm
    2. print perfect pass rate: which is if column 'res' == 'same' /30
    3. print delivery rate: which is if column 'res' != 'missing_output' /30
    4. get precision, recall, f1


to get precision, recall, f1, 


In [2]:
import os
import pandas as pd
import shutil

# Load Excel sheet
path = "/Users/mihokoda/Desktop/City LLM final analysis/book1.xlsx"
df = pd.read_excel(path, sheet_name="dubai")

# Only keep rows where res == 'same' and llm_name is one of the targets
df_filtered = df[(df['res'] == 'same') & (df['llm_name'].isin(['react', 'reflexion']))]

# Process each row
for _, row in df_filtered.iterrows():
    test_case = row['test_case']  # e.g., tc_dubai_res_sim_6
    llm_name = row['llm_name']    # 'react' or 'react + reflexion'

    # Parse difficulty and index
    parts = test_case.split('_')
    difficulty = parts[3]  # 'sim', 'med', or 'hard'
    case_index = parts[-1]  # e.g., '6'

    # File paths
    case_dir = f"/Users/mihokoda/Desktop/LocationReasoner/dubai_res/{difficulty}/{test_case}"
    old_csv = os.path.join(case_dir, f"test_case_{case_index}_{llm_name}.csv")
    objective_csv = os.path.join(case_dir, "objective.csv")

    # Replace file
    if os.path.exists(objective_csv):
        try:
            os.remove(old_csv)  # delete the old prediction file if it exists
        except FileNotFoundError:
            pass  # no file to remove, that's fine
        shutil.copy(objective_csv, old_csv)  # copy objective.csv as the new result file
        print(f"✅ Replaced {old_csv} with objective.csv")
    else:
        print(f"❌ objective.csv not found for {test_case}")


✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_0/test_case_0_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_2/test_case_2_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_3/test_case_3_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_6/test_case_6_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/sim/tc_dubai_res_sim_7/test_case_7_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/med/tc_dubai_res_med_4/test_case_4_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/med/tc_dubai_res_med_8/test_case_8_react.csv with objective.csv
✅ Replaced /Users/mihokoda/Desktop/LocationReasoner/dubai_res/hard/tc_dubai_res_hard_3/test_case_3_react.csv with obje

In [ ]:
import os
import pandas as pd

base_dir = "/Users/mihokoda/Desktop/LocationReasoner/dubai_res/"
difficulties = ["sim", "med", "hard"]
model_filename = "gemini2.5.csv"
output_path = "/Users/mihokoda/Desktop/LocationReasoner/gemini2.5_eval.xlsx"

results = []

for difficulty in difficulties:
    for i in range(10):  # tc_dubai_res_sim_0 to tc_dubai_res_sim_9
        test_case = f"tc_dubai_res_{difficulty}_{i}"
        folder = os.path.join(base_dir, difficulty, test_case)

        if not os.path.isdir(folder):
            print(f"❌ Skipping missing folder: {folder}")
            continue

        obj_path = os.path.join(folder, "objective.csv")
        pred_path = os.path.join(folder, model_filename)

        delivered = os.path.exists(pred_path)

        # Load objective
        if os.path.exists(obj_path):
            try:
                obj_df = pd.read_csv(obj_path)
                y_true = set(obj_df["Location"].dropna().astype(str)) if "Location" in obj_df else set()
            except Exception:
                y_true = set()
        else:
            y_true = set()

        # Load prediction
        if delivered:
            try:
                pred_df = pd.read_csv(pred_path)
                y_pred = set(pred_df["Location"].dropna().astype(str)) if "Location" in pred_df else set()
            except Exception:
                y_pred = set()
        else:
            y_pred = set()

        # Compute metrics
        if not y_true and not y_pred:
            precision = recall = f1 = 1.0
        else:
            tp = len(y_true & y_pred)
            fp = len(y_pred - y_true)
            fn = len(y_true - y_pred)
            precision = tp / (tp + fp) if tp + fp > 0 else 0.0
            recall = tp / (tp + fn) if tp + fn > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

        perfect_pass = y_true == y_pred

        results.append({
            "difficulty": difficulty,
            "test_case": test_case,
            "delivered": delivered,
            "perfect_pass": perfect_pass,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

# Create DataFrame
results_df = pd.DataFrame(results)

# Aggregate summary
summary = []
for level in difficulties + ["overall"]:
    df = results_df if level == "overall" else results_df[results_df["difficulty"] == level]

    delivery_rate = df["delivered"].mean()
    perfect_rate = df["perfect_pass"].mean()
    precision = df["precision"].mean()
    recall = df["recall"].mean()
    f1 = df["f1_score"].mean()

    summary.append({
        "Difficulty": level,
        "Delivery Rate": f"{delivery_rate:.2%}",
        "Perfect Pass": f"{perfect_rate:.2%}",
        "Precision": round(precision, 2),
        "Recall": round(recall, 2),
        "F1 Score": round(f1, 2)
    })

summary_df = pd.DataFrame(summary)

# Save both to Excel
with pd.ExcelWriter(output_path) as writer:
    results_df.to_excel(writer, index=False, sheet_name="Test Cases")
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

print(f"✅ Evaluation saved to {output_path}")


✅ Evaluation saved to /Users/mihokoda/Desktop/LocationReasoner/gemini2.5_eval.xlsx


In [4]:
import os
import pandas as pd

base_dir = "/Users/mihokoda/Desktop/LocationReasoner/dubai_res/"
difficulties = ["sim", "med", "hard"]
models = ["react", "reflexion"]  # corresponds to file suffixes
output_path = "/Users/mihokoda/Desktop/LocationReasoner/react_reflexion_eval.xlsx"

results = []

for model in models:
    for difficulty in difficulties:
        for i in range(10):  # tc_dubai_res_sim_0 to tc_dubai_res_sim_9
            test_case = f"tc_dubai_res_{difficulty}_{i}"
            folder = os.path.join(base_dir, difficulty, test_case)

            if not os.path.isdir(folder):
                print(f"❌ Skipping missing folder: {folder}")
                continue

            obj_path = os.path.join(folder, "objective.csv")
            pred_path = os.path.join(folder, f"test_case_{i}_{model}.csv")

            delivered = os.path.exists(pred_path)

            # Load objective
            if os.path.exists(obj_path):
                try:
                    obj_df = pd.read_csv(obj_path)
                    y_true = set(obj_df["Location"].dropna().astype(str)) if "Location" in obj_df else set()
                except Exception:
                    y_true = set()
            else:
                y_true = set()

            # Load prediction
            if delivered:
                try:
                    pred_df = pd.read_csv(pred_path)
                    y_pred = set(pred_df["Location"].dropna().astype(str)) if "Location" in pred_df else set()
                except Exception:
                    y_pred = set()
            else:
                y_pred = set()

            # Compute metrics
            if not y_true and not y_pred:
                precision = recall = f1 = 1.0
            else:
                tp = len(y_true & y_pred)
                fp = len(y_pred - y_true)
                fn = len(y_true - y_pred)
                precision = tp / (tp + fp) if tp + fp > 0 else 0.0
                recall = tp / (tp + fn) if tp + fn > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

            perfect_pass = y_true == y_pred

            results.append({
                "model": model,
                "difficulty": difficulty,
                "test_case": test_case,
                "delivered": delivered,
                "perfect_pass": perfect_pass,
                "precision": precision,
                "recall": recall,
                "f1_score": f1
            })

# Create results DataFrame
results_df = pd.DataFrame(results)

# Compute summary
summary = []
for model in models:
    for level in difficulties + ["overall"]:
        df = results_df[(results_df["model"] == model)] if level == "overall" else \
             results_df[(results_df["model"] == model) & (results_df["difficulty"] == level)]

        delivery_rate = df["delivered"].mean()
        perfect_rate = df["perfect_pass"].mean()
        precision = df["precision"].mean()
        recall = df["recall"].mean()
        f1 = df["f1_score"].mean()

        summary.append({
            "Model": f"React + Reflexion" if model == "reflexion" else "React",
            "Difficulty": level,
            "Delivery Rate": f"{delivery_rate:.2%}",
            "Perfect Pass": f"{perfect_rate:.2%}",
            "Precision": round(precision, 2),
            "Recall": round(recall, 2),
            "F1 Score": round(f1, 2)
        })

summary_df = pd.DataFrame(summary)

# Save to Excel
with pd.ExcelWriter(output_path) as writer:
    results_df.to_excel(writer, index=False, sheet_name="Test Cases")
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

print(f"✅ Evaluation saved to {output_path}")


✅ Evaluation saved to /Users/mihokoda/Desktop/LocationReasoner/react_reflexion_eval.xlsx
